In [ ]:
!pip install evaluate seqeval torch -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


In [ ]:
!gdown 1Oj8EvhX0wuvr_J5sLzr3VjtFR94GtVAC

Downloading...
From: https://drive.google.com/uc?id=1Oj8EvhX0wuvr_J5sLzr3VjtFR94GtVAC
To: /content/PBP-ud-complete.conllu
100% 11.0M/11.0M [00:00<00:00, 116MB/s]


In [ ]:
import json
import torch
from torch import nn
import random
import numpy as np
from collections import defaultdict, Counter
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)
from datasets import Dataset
from itertools import chain

In [ ]:
def parse_propbank_br_conllu(filepath):
    sentences = []
    current_tokens = []
    current_rows = []

    # Lê e separa por sentenças
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_rows:
                    sentences.append(current_rows)
                    current_rows = []
                continue
            if "-" in line.split("\t")[0]:
                continue  # ignora intervalos tipo 13-14
            parts = line.split("\t")
            if len(parts) < 10:
                continue
            current_rows.append(parts)
        if current_rows:
            sentences.append(current_rows)

    parsed_semantics = []
    flat_instances = []
    sentence_indexer = 0

    # Processa cada sentença
    for rows in sentences:
        tokens = [r[1] for r in rows]
        preds = {}
        args_by_pred = {}

        # 1️- Identifica predicados
        for r in rows:
            idx, form, lemma, upos, _, feats, head, deprel, pred, role = r
            if pred != "_" and pred.strip():
                preds[int(idx)] = pred
                args_by_pred[int(idx)] = {}

        # 2️-  Coleta papéis ligados a cada predicado
        for r in rows:
            idx, form, lemma, upos, _, feats, head, deprel, pred, role = r
            if role != "_" and role.strip():
                for part in role.split("|"):
                    if ":" not in part:
                        continue
                    role_label, pred_id = part.split(":")
                    if pred_id.isdigit() and int(pred_id) in args_by_pred:
                        args_by_pred[int(pred_id)].setdefault(role_label, []).append(form)

        # 3️ - Estrutura rica
        predicate_entries = []
        for pred_id, frame in preds.items():
            predicate_entries.append({
                "id": pred_id,
                "predicate": frame,
                "roles": args_by_pred.get(pred_id, {})
            })
        parsed_semantics.append({
            "sentence": tokens,
            "predicates": predicate_entries
        })

        # 4️- Estrutura achatada pra treino
        for pred_id, frame in preds.items():
            labels = []
            for r in rows:
                form = r[1]
                role_str = r[9]
                label = "O"
                if str(pred_id) == r[0]:
                    label = "PRED"
                elif role_str != "_":
                    for part in role_str.split("|"):
                        if f":{pred_id}" in part:
                            label = part.split(":")[0].upper()
                            break
                labels.append(label)
            flat_instances.append({
                "id": sentence_indexer,
                "predicate": frame,
                "tokens": tokens,
                "labels": labels
            })
        sentence_indexer += 1

    return parsed_semantics, flat_instances

parsed_semantics, flat_instances = parse_propbank_br_conllu("PBP-ud-complete.conllu")

# Salva as duas versões em JSON
with open("propbank_br_structured.json", "w", encoding="utf-8") as f:
    json.dump(parsed_semantics, f, ensure_ascii=False, indent=2)

with open("propbank_br_flat.json", "w", encoding="utf-8") as f:
    json.dump(flat_instances, f, ensure_ascii=False, indent=2)

print(f"Total de sentenças: {len(parsed_semantics)}")
print(f"Total de instâncias SRL (achatadas): {len(flat_instances)}")

Total de sentenças: 8418
Total de instâncias SRL (achatadas): 14229


In [ ]:
for i in range(len(parsed_semantics)):
  if i == 2: break
  print('\n'+"*"*10+'\n')
  print(parsed_semantics[i]['sentence'])
  print(parsed_semantics[i]['predicates'])


**********

['Lula', 'diz', 'que', 'está', "'", 'lascado', "'", ',', 'mas', 'que', 'ainda', 'tem', 'força', 'como', 'cabo', 'eleitoral', '.']
[{'id': 2, 'predicate': 'dizer.01', 'roles': {'Arg0': ['Lula'], 'Arg1': ['lascado']}}, {'id': 4, 'predicate': 'estar.301', 'roles': {}}]

**********

['Eu', 'sei', 'que', 'tô', 'lascado', ',', 'todo', 'dia', 'tem', 'um', 'processo', '.']
[{'id': 2, 'predicate': 'saber.01', 'roles': {'Arg0': ['Eu'], 'Arg1': ['lascado']}}, {'id': 4, 'predicate': 'estar.301', 'roles': {}}, {'id': 9, 'predicate': 'ter.02', 'roles': {'ArgM-tmp': ['dia'], 'Arg1': ['processo']}}]


In [ ]:
for i in range(len(flat_instances)):
  if i == 5: break
  print('\n'+"*"*10+'\n')
  print(flat_instances[i]['id'])
  print(flat_instances[i]['predicate'])
  print(flat_instances[i]['tokens'])
  print(flat_instances[i]['labels'])


**********

0
dizer.01
['Lula', 'diz', 'que', 'está', "'", 'lascado', "'", ',', 'mas', 'que', 'ainda', 'tem', 'força', 'como', 'cabo', 'eleitoral', '.']
['ARG0', 'PRED', 'O', 'O', 'O', 'ARG1', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

**********

0
estar.301
['Lula', 'diz', 'que', 'está', "'", 'lascado', "'", ',', 'mas', 'que', 'ainda', 'tem', 'força', 'como', 'cabo', 'eleitoral', '.']
['O', 'O', 'O', 'PRED', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

**********

1
saber.01
['Eu', 'sei', 'que', 'tô', 'lascado', ',', 'todo', 'dia', 'tem', 'um', 'processo', '.']
['ARG0', 'PRED', 'O', 'O', 'ARG1', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

**********

1
estar.301
['Eu', 'sei', 'que', 'tô', 'lascado', ',', 'todo', 'dia', 'tem', 'um', 'processo', '.']
['O', 'O', 'O', 'PRED', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

**********

1
ter.02
['Eu', 'sei', 'que', 'tô', 'lascado', ',', 'todo', 'dia', 'tem', 'um', 'processo', '.']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'A

In [ ]:
random.seed(42)
set_seed(42)

# Agrupando por sentenças
groups = defaultdict(list)
for inst in flat_instances:
    groups[inst["id"]].append(inst)

predicates = list(groups.keys())
random.shuffle(predicates)

# Divisão 70/10/20 e criação de conjuntos
train_ratio = 0.7
valid_ratio = 0.1
split_idx_train_to_valid = int(len(predicates) * train_ratio)
split_idx_valid_to_test = int(len(predicates) * (train_ratio + valid_ratio))

train_preds = predicates[:split_idx_train_to_valid]
valid_preds = predicates[split_idx_train_to_valid:split_idx_valid_to_test]
test_preds = predicates[split_idx_valid_to_test:]

train_instances = [inst for p in train_preds for inst in groups[p]]
valid_instances = [inst for p in valid_preds for inst in groups[p]]
test_instances = [inst for p in test_preds for inst in groups[p]]

In [ ]:
def analyze_dataset(instances, name="Dataset"):
    """Análise detalhada do dataset"""
    print(f"\n{'='*60}")
    print(f"Análise: {name}")
    print(f"{'='*60}")

    # Agrupar por sentença
    sentences = defaultdict(list)
    for inst in instances:
        sentences[inst["id"]].append(inst)

    print(f"Total de instâncias: {len(instances)}")
    print(f"Total de sentenças únicas: {len(sentences)}")
    print(f"Média de predicados por sentença: {len(instances)/len(sentences):.2f}")

    # Distribuição de labels
    all_labels = list(chain.from_iterable([inst["labels"] for inst in instances]))
    label_counts = Counter(all_labels)

    print(f"\nDistribuição de Labels:")
    total_labels = len(all_labels)
    for label, count in label_counts.most_common():
        pct = (count/total_labels)*100
        print(f"  {label:15s}: {count:6d} ({pct:5.2f}%)")

    # Tamanho das sentenças
    sentence_lengths = [len(inst["tokens"]) for inst in instances]
    print(f"\nTamanho das Sentenças:")
    print(f"  Mínimo: {min(sentence_lengths)}")
    print(f"  Máximo: {max(sentence_lengths)}")
    print(f"  Média: {np.mean(sentence_lengths):.2f}")
    print(f"  Mediana: {np.median(sentence_lengths):.2f}")

    # Predicados mais comuns
    predicates = [inst["predicate"] for inst in instances]
    pred_counts = Counter(predicates)
    print(f"\nTop 10 Predicados:")
    for pred, count in pred_counts.most_common(10):
        print(f"  {pred:20s}: {count:4d}")

    return label_counts

In [ ]:
def check_data_leakage(train_instances, valid_instances, test_instances):
    """Verifica se há vazamento de sentenças entre os conjuntos"""
    train_ids = set([inst["id"] for inst in train_instances])
    valid_ids = set([inst["id"] for inst in valid_instances])
    test_ids = set([inst["id"] for inst in test_instances])

    print(f"\n{'='*60}")
    print("Verificação de Vazamento de Dados")
    print(f"{'='*60}")

    train_valid_overlap = train_ids & valid_ids
    train_test_overlap = train_ids & test_ids
    valid_test_overlap = valid_ids & test_ids

    if train_valid_overlap:
        print(f"{len(train_valid_overlap)} sentenças compartilhadas entre treino e validação!")
    else:
        print("Nenhuma sentença compartilhada entre treino e validação")

    if train_test_overlap:
        print(f"{len(train_test_overlap)} sentenças compartilhadas entre treino e teste!")
    else:
        print("Nenhuma sentença compartilhada entre treino e teste")

    if valid_test_overlap:
        print(f"{len(valid_test_overlap)} sentenças compartilhadas entre validação e teste!")
    else:
        print("Nenhuma sentença compartilhada entre validação e teste")


In [ ]:
# ANÁLISE DOS DADOS
train_label_counts = analyze_dataset(train_instances, "Treino")
valid_label_counts = analyze_dataset(valid_instances, "Validação")
test_label_counts = analyze_dataset(test_instances, "Teste")
check_data_leakage(train_instances, valid_instances, test_instances)

# Criar datasets
train_dataset = Dataset.from_dict({
    "predicate": [x["predicate"] for x in train_instances],
    "tokens": [x["tokens"] for x in train_instances],
    "labels": [x["labels"] for x in train_instances],
})

valid_dataset = Dataset.from_dict({
    "predicate": [x["predicate"] for x in valid_instances],
    "tokens": [x["tokens"] for x in valid_instances],
    "labels": [x["labels"] for x in valid_instances],
})

test_dataset = Dataset.from_dict({
    "predicate": [x["predicate"] for x in test_instances],
    "tokens": [x["tokens"] for x in test_instances],
    "labels": [x["labels"] for x in test_instances],
})


Análise: Treino
Total de instâncias: 9857
Total de sentenças únicas: 4998
Média de predicados por sentença: 1.97

Distribuição de Labels:
  O              : 186782 (87.16%)
  PRED           :   9857 ( 4.60%)
  ARG1           :   6068 ( 2.83%)
  ARG0           :   3991 ( 1.86%)
  ARG2           :   1523 ( 0.71%)
  ARGM-TMP       :   1177 ( 0.55%)
  ARG1_D         :    687 ( 0.32%)
  ARGM-MNR       :    652 ( 0.30%)
  ARGM-LOC       :    592 ( 0.28%)
  ARGM-ADV       :    542 ( 0.25%)
  ARGM-NEG       :    478 ( 0.22%)
  ARG0_D         :    467 ( 0.22%)
  ARGM-DIS       :    330 ( 0.15%)
  ARGM-CAU       :    196 ( 0.09%)
  ARG3           :    193 ( 0.09%)
  ARGM-PRP       :    165 ( 0.08%)
  ARGM-EXT       :    116 ( 0.05%)
  ARGM-NSE       :    109 ( 0.05%)
  ARG4           :     94 ( 0.04%)
  ARGM-SRC       :     93 ( 0.04%)
  ARGM-PRD       :     67 ( 0.03%)
  ARGM-COND      :     57 ( 0.03%)
  ARGM-COM       :     22 ( 0.01%)
  ARGM-CONSEQ    :     14 ( 0.01%)
  INC            :   

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoConfig

class SRLWithPredicateIndicator(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        hidden = self.config.hidden_size

        # Camada para transformar o bit do predicado em embedding
        self.predicate_embedding = nn.Linear(1, hidden)

        # Classificador
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids, attention_mask, predicate_indicator, labels=None, **kwargs):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        sequence_output = outputs.last_hidden_state  # [B, T, H]

        # Garantindo que predicate_indicator tem valores
        if torch.sum(predicate_indicator) == 0:
            print("Aviso: predicate_indicator está zerado!")

        # Adiciona embedding do predicado
        pred_embed = self.predicate_embedding(predicate_indicator.unsqueeze(-1))
        sequence_output = sequence_output + pred_embed

        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(
                logits.view(-1, logits.shape[-1]),
                labels.view(-1)
            )

        return {"loss": loss, "logits": logits}

In [ ]:
def verify_predicate_indicators(tokenized_dataset, num_samples=5):
    """
    Verifica se os predicate_indicators estão corretos
    """
    print("\n" + "="*60)
    print("VERIFICAÇÃO DOS PREDICATE INDICATORS")
    print("="*60)

    for i in range(min(num_samples, len(tokenized_dataset))):
        example = tokenized_dataset[i]

        tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
        pred_indicators = example["predicate_indicator"]
        labels = example["labels"]

        # Encontrar onde está o predicado
        pred_positions = [j for j, p in enumerate(pred_indicators) if p == 1]

        print(f"\nExemplo {i+1}:")
        print(f"  Predicados em posições: {pred_positions}")

        if pred_positions:
            for pos in pred_positions:
                if pos < len(tokens):
                    print(f"    Posição {pos}: '{tokens[pos]}' (label_id={labels[pos]})")
        else:
            print("  NENHUM PREDICADO ENCONTRADO!")

        # Mostrar contexto
        print("  Contexto:")
        for j, (tok, pred, lab) in enumerate(zip(tokens[:20], pred_indicators[:20], labels[:20])):
            marker = "X" if pred == 1 else "  "
            label_name = label_list[lab] if lab != -100 else "PAD"
            print(f"    {marker} {j:2d}: {tok:15s} pred={pred} label={label_name}")

In [ ]:
# Tokenizador
model_name = "neuralmind/bert-large-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Gerando lista de labels
all_labels = list(chain.from_iterable(train_dataset["labels"]))
label_list = sorted(list(set(all_labels)))

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print(f"\n{'='*60}")
print(f"Labels identificados: {label_list}")
print(f"Total de labels: {len(label_list)}")
print(f"{'='*60}\n")

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=80,
    )

    all_labels = []
    all_pred_indicators = []

    for i, (labels, tokens) in enumerate(zip(examples["labels"], examples["tokens"])):
        word_ids = tokenized.word_ids(batch_index=i)

        # Encontrando o índice do predicado nos LABELS
        predicate_index = None
        for idx, label in enumerate(labels):
            if label == "PRED":
                predicate_index = idx
                break

        if predicate_index is None:
            # Se não encontrar predicado, algo está errado
            print(f"Aviso: Nenhum predicado encontrado em: {tokens}")
            predicate_index = -1

        aligned_labels = []
        aligned_predicates = []
        prev_word = None

        for word_id in word_ids:
            if word_id is None:
                # Tokens especiais ([CLS], [SEP], [PAD])
                aligned_labels.append(-100)
                aligned_predicates.append(0)
                continue

            # Primeira subpalavra
            if word_id != prev_word:
                aligned_labels.append(label2id[labels[word_id]])
                # Comparando word_id com predicate_index
                aligned_predicates.append(1 if word_id == predicate_index else 0)
            else:
                # Subpalavras seguintes
                aligned_labels.append(-100)
                aligned_predicates.append(0)

            prev_word = word_id

        all_labels.append(aligned_labels)
        all_pred_indicators.append(aligned_predicates)

    tokenized["labels"] = all_labels
    tokenized["predicate_indicator"] = all_pred_indicators

    return tokenized


train_tokenized_datasets = train_dataset.map(tokenize_and_align_labels, batched=True)
valid_tokenized_datasets = valid_dataset.map(tokenize_and_align_labels, batched=True)

print("\nVERIFICANDO TRAIN DATASET:")
verify_predicate_indicators(train_tokenized_datasets, num_samples=3)

print("\nVERIFICANDO VALID DATASET:")
verify_predicate_indicators(valid_tokenized_datasets, num_samples=3)

#  Modelo
model = SRLWithPredicateIndicator(model_name, len(label_list))

# Collator
def data_collator(features):
    """
    VERSÃO CORRIGIDA que garante passar predicate_indicator
    """
    batch = {}

    batch["input_ids"] = torch.tensor([f["input_ids"] for f in features])
    batch["attention_mask"] = torch.tensor([f["attention_mask"] for f in features])
    batch["labels"] = torch.tensor([f["labels"] for f in features])

    # Garantindo que predicate_indicator está presente
    if "predicate_indicator" not in features[0]:
        raise ValueError("predicate_indicator não está no dataset!")

    batch["predicate_indicator"] = torch.tensor(
        [f["predicate_indicator"] for f in features]
    ).float()

    # Checando se tem predicados
    pred_sum = batch["predicate_indicator"].sum()
    if pred_sum == 0:
        print("Aviso: Batch sem nenhum predicado!")

    return batch


Labels identificados: ['ARG0', 'ARG0_D', 'ARG1', 'ARG1_D', 'ARG2', 'ARG3', 'ARG4', 'ARGM-ADV', 'ARGM-CAU', 'ARGM-COM', 'ARGM-COMP', 'ARGM-COND', 'ARGM-CONSEQ', 'ARGM-DIR', 'ARGM-DIS', 'ARGM-EXT', 'ARGM-LOC', 'ARGM-MNR', 'ARGM-NEG', 'ARGM-NSE', 'ARGM-PRD', 'ARGM-PRP', 'ARGM-SRC', 'ARGM-TMP', 'INC', 'O', 'PRED']
Total de labels: 27



Map:   0%|          | 0/9857 [00:00<?, ? examples/s]

Map:   0%|          | 0/1422 [00:00<?, ? examples/s]


VERIFICANDO TRAIN DATASET:

VERIFICAÇÃO DOS PREDICATE INDICATORS

Exemplo 1:
  Predicados em posições: [3]
    Posição 3: 'viajar' (label_id=26)
  Contexto:
        0: [CLS]           pred=0 label=PAD
        1: A               pred=0 label=O
        2: o               pred=0 label=O
    X  3: viajar          pred=1 label=PRED
        4: sozinha         pred=0 label=ARGM-COM
        5: ,               pred=0 label=O
        6: estaria         pred=0 label=O
        7: mais            pred=0 label=O
        8: vulner          pred=0 label=O
        9: ##ável          pred=0 label=PAD
       10: a               pred=0 label=O
       11: crimes          pred=0 label=O
       12: ,               pred=0 label=O
       13: como            pred=0 label=O
       14: fur             pred=0 label=O
       15: ##tos           pred=0 label=PAD
       16: .               pred=0 label=O
       17: [SEP]           pred=0 label=PAD
       18: [PAD]           pred=0 label=PAD
       19: [PAD]         

In [ ]:
# Métrica de avaliação
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)

    # Remover -100 e converter IDs em strings
    true_preds = []
    true_labels = []

    for pred_seq, lab_seq in zip(predictions, labels):
        seq_pred = []
        seq_gold = []
        for p, l in zip(pred_seq, lab_seq):
            if l == -100:
                continue
            seq_pred.append(label_list[p])
            seq_gold.append(label_list[l])
        true_preds.append(seq_pred)
        true_labels.append(seq_gold)

    return span_f1(true_preds, true_labels)


def extract_spans(label_seq, sentence_id):
    """
    Extrai spans com informação da sentença para comparação correta
    Retorna: [(sentence_id, label, start, end), ...]
    """
    spans = []
    current_label = None
    start = None

    for i, label in enumerate(label_seq):
        if label == "O":
            if current_label is not None:
                spans.append((sentence_id, current_label, start, i - 1))
                current_label = None
            continue

        if label != current_label:
            if current_label is not None:
                spans.append((sentence_id, current_label, start, i - 1))
            current_label = label
            start = i

    if current_label is not None:
        spans.append((sentence_id, current_label, start, len(label_seq) - 1))

    return spans

def span_f1(pred_seqs, gold_seqs):
    """
    Calcula Span F1 corretamente, preservando contexto de sentença
    """
    pred_spans = []
    gold_spans = []

    for sent_id, (pred, gold) in enumerate(zip(pred_seqs, gold_seqs)):
        pred_spans.extend(extract_spans(pred, sent_id))
        gold_spans.extend(extract_spans(gold, sent_id))

    pred_set = set(pred_spans)
    gold_set = set(gold_spans)

    tp = len(pred_set & gold_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {
        "span_precision": precision,
        "span_recall": recall,
        "span_f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/srl-bert-model_2"

Mounted at /content/drive


In [ ]:
args = TrainingArguments(
    output_dir="./srl-bert",
    num_train_epochs=6,
    learning_rate=2e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    warmup_ratio=0.1,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    logging_steps=200,
    logging_first_step=True,

    metric_for_best_model="span_f1",
    greater_is_better=True,
    load_best_model_at_end=True,

    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tokenized_datasets,
    eval_dataset=valid_tokenized_datasets,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],

)

trainer.train()

/tmp/ipython-input-2083658387.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Span Precision,Span Recall,Span F1,Tp,Fp,Fn
1,7.753100,0.315721,0.958462,0.321051,0.480988,1246,54,2635


Epoch,Training Loss,Validation Loss,Span Precision,Span Recall,Span F1,Tp,Fp,Fn
1,7.753100,0.315721,0.958462,0.321051,0.480988,1246,54,2635
2,1.762400,0.226541,0.782674,0.535429,0.635863,2078,577,1803
3,0.473500,0.206770,0.717957,0.597526,0.652229,2319,911,1562
4,0.394100,0.193068,0.740375,0.599588,0.662585,2327,816,1554
5,0.394100,0.190124,0.743346,0.604483,0.666761,2346,810,1535
6,0.351400,0.189726,0.746411,0.602937,0.667047,2340,795,1541


TrainOutput(global_step=930, training_loss=0.6945806493041335, metrics={'train_runtime': 2718.7734, 'train_samples_per_second': 21.753, 'train_steps_per_second': 0.342, 'total_flos': 0.0, 'train_loss': 0.6945806493041335, 'epoch': 6.0})

In [ ]:
trainer.save_model(save_dir)          # salva o modelo e config
tokenizer.save_pretrained(save_dir)  # salva o tokenizer

import json
with open(f"{save_dir}/training_metrics.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)

# pra carregar seria algo tipo isso
# from transformers import AutoTokenizer, AutoModelForTokenClassification

# model = AutoModelForTokenClassification.from_pretrained(save_dir)
# tokenizer = AutoTokenizer.from_pretrained(save_dir)

In [ ]:
test_tokenized = test_dataset.map(tokenize_and_align_labels, batched=True)
test_results = trainer.predict(test_tokenized)

print(f"\n{'='*60}")
print("RESULTADOS FINAIS NO CONJUNTO DE TESTE")
print(f"{'='*60}")
for metric, value in test_results.metrics.items():
    print(f"{metric}: {value:.4f}")

Map:   0%|          | 0/2950 [00:00<?, ? examples/s]


RESULTADOS FINAIS NO CONJUNTO DE TESTE
test_loss: 0.1986
test_span_precision: 0.7269
test_span_recall: 0.6026
test_span_f1: 0.6590
test_tp: 5011.0000
test_fp: 1883.0000
test_fn: 3304.0000
test_runtime: 41.1282
test_samples_per_second: 71.7270
test_steps_per_second: 8.9720
